In [1]:
import numpy as np
import sklearn
import joblib
print(f"numpy: {np.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"joblib: {joblib.__version__}")

numpy: 1.24.3
sklearn: 1.4.2
joblib: 1.4.2


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import os

os.makedirs('C:/data/BrandPulse-AI/deployment/models', exist_ok=True)

df = pd.read_csv('C:/data/BrandPulse-AI/data/processed/cleaned_tweets.csv')
df = df.dropna(subset=['cleaned_text'])
df = df[df['cleaned_text'].str.strip() != '']

X = df['cleaned_text'].values
y = df['polarity'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Fitting TF-IDF...")
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)

print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs')
lr_model.fit(X_train_tfidf, y_train)

joblib.dump(tfidf,    'C:/data/BrandPulse-AI/deployment/models/tfidf_vectorizer.pkl', compress=3)
joblib.dump(lr_model, 'C:/data/BrandPulse-AI/deployment/models/logistic_model.pkl', compress=3)

print(f" Saved with numpy {np.__version__} sklearn {sklearn.__version__}")

Fitting TF-IDF...
Training Logistic Regression...
 Saved with numpy 1.24.3 sklearn 1.4.2


In [12]:
import shutil
import os

# Copy tokenizer only
shutil.copy(
    'C:/data/BrandPulse-AI/models/tokenizer.json',
    'C:/data/BrandPulse-AI/deployment/models/tokenizer.json'
)
print("Tokenizer copied ")

# Verify all files
print("\nDeployment models:")
for f in os.listdir('C:/data/BrandPulse-AI/deployment/models'):
    size = os.path.getsize(f'C:/data/BrandPulse-AI/deployment/models/{f}')
    print(f"  {f}: {size/1024/1024:.1f} MB")

Tokenizer copied 

Deployment models:
  logistic_model.pkl: 0.4 MB
  lstm_best.h5: 30.6 MB
  tfidf_vectorizer.pkl: 26.5 MB
  tokenizer.json: 33.6 MB


In [6]:
requirements = """streamlit
tensorflow==2.13.0
scikit-learn==1.4.2
pandas
numpy==1.24.3
plotly
joblib==1.4.2
nltk
"""

with open('C:/data/BrandPulse-AI/deployment/requirements.txt', 'w') as f:
    f.write(requirements)
print("requirements.txt created ")

requirements.txt created 


In [7]:
with open('C:/data/BrandPulse-AI/deployment/.python-version', 'w') as f:
    f.write('3.11')
print(".python-version created ")

.python-version created 


In [10]:
with open('C:/data/BrandPulse-AI/app/app.py', 'r', encoding='utf-8') as f:
    content = f.read()

# Fix 1: Replace local imports with inline clean_tweet
content = content.replace(
    "sys.path.append('C:/data/BrandPulse-AI')\nfrom src.preprocessing import clean_tweet",
    """import re
import os
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english')) - {'not', 'no', 'nor', 'never'}
def clean_tweet(text):
    text = text.lower()
    text = re.sub(r'http\\S+|www\\S+|https\\S+', '', text)
    text = re.sub(r'@\\w+', '', text)
    text = re.sub(r'#\\w+', '', text)
    text = re.sub(r'\\d+', '', text)
    text = re.sub(r'[^\\w\\s]', '', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return ' '.join([w for w in tokens if w not in STOP_WORDS])"""
)

# Fix 2: Replace hardcoded model paths
content = content.replace(
    """    tfidf    = joblib.load('C:/data/BrandPulse-AI/models/tfidf_vectorizer.pkl')
    lr_model = joblib.load('C:/data/BrandPulse-AI/models/logistic_model.pkl')
    lstm     = load_model('C:/data/BrandPulse-AI/models/lstm_best.h5')
    with open('C:/data/BrandPulse-AI/models/tokenizer.json') as f:""",
    """    BASE = os.path.dirname(os.path.abspath(__file__))
    MODELS = os.path.join(BASE, 'models')
    tfidf    = joblib.load(os.path.join(MODELS, 'tfidf_vectorizer.pkl'))
    lr_model = joblib.load(os.path.join(MODELS, 'logistic_model.pkl'))
    lstm     = load_model(os.path.join(MODELS, 'lstm_best.h5'))
    with open(os.path.join(MODELS, 'tokenizer.json')) as f:"""
)

# Save to deployment folder
with open('C:/data/BrandPulse-AI/deployment/app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("deployment/app.py fixed ")

deployment/app.py fixed 
